# CarMaker Segmentation Training on Google Colab

이 노트북은 Google Drive에 올려둔 `carmaker_image/data` 형식의 PNG dataset으로 `src/segmentation/train.py`를 실행한다.

Colab 메뉴에서 `Runtime > Change runtime type > GPU`를 먼저 선택한다.

In [ ]:
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

## 1. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 2. Colab Settings

`DRIVE_PROJECT_DIR` 아래에 `data/raw_images`, `data/gt_post_processed`, `data/csv/manifest.csv`가 있어야 한다.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/2025-AILAB-WINTER-INTERNSHIP/carmaker.git"
BRANCH = "main"
WORK_DIR = Path("/content/carmaker")

DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/carmaker_segmentation")
DATA_ROOT = DRIVE_PROJECT_DIR / "data"
MANIFEST = DATA_ROOT / "csv" / "manifest.csv"
RUN_DIR = DRIVE_PROJECT_DIR / "runs"
DEBUG_DIR = DRIVE_PROJECT_DIR / "debug_dataset"

IMAGE_SIZE = [1920, 1080]
EPOCHS = 16
BATCH_SIZE = 1
NUM_WORKERS = 4
BASE_CHANNELS = 32
LOSS = "focal"

RUN_DIR.mkdir(parents=True, exist_ok=True)
DEBUG_DIR.mkdir(parents=True, exist_ok=True)
print("DATA_ROOT:", DATA_ROOT)
print("RUN_DIR:", RUN_DIR)
print("run naming:", "{loss}_ep{epochs}_{timestamp}")

## 3. Clone or Update Repository

In [ ]:
import os
import subprocess

if (WORK_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(WORK_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(WORK_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(WORK_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(WORK_DIR)], check=True)

os.chdir(WORK_DIR)

TRAIN_ENV = os.environ.copy()
TRAIN_ENV["PYTHONUNBUFFERED"] = "1"
TRAIN_ENV["TQDM_POSITION"] = "-1"
TRAIN_ENV["TQDM_MININTERVAL"] = "1"

print("cwd:", Path.cwd())

## 4. Install Python Dependencies

In [ ]:
%pip -q install opencv-python-headless pyyaml tensorboard tqdm "protobuf>=5.29.1,<6"

## 5. Check Dataset

In [ ]:
required_paths = [
    DATA_ROOT,
    DATA_ROOT / "raw_images",
    DATA_ROOT / "gt_post_processed",
]

for path in required_paths:
    print(path, "OK" if path.exists() else "MISSING")

print(MANIFEST, "OK" if MANIFEST.exists() else "MISSING")
print("raw png count:", len(list((DATA_ROOT / "raw_images").rglob("*.png"))) if (DATA_ROOT / "raw_images").exists() else 0)
print("mask png count:", len(list((DATA_ROOT / "gt_post_processed").rglob("*.png"))) if (DATA_ROOT / "gt_post_processed").exists() else 0)

missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError("Missing dataset paths: " + ", ".join(missing))

## 6. Generate Colab Config

In [ ]:
import yaml

CONFIG_DIR = Path("/content/carmaker_colab")
CONFIG_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = CONFIG_DIR / "segmentation_unet_colab.generated.yaml"

config = {
    "data_root": str(DATA_ROOT),
    "manifest": str(MANIFEST),
    "cameras": "front,left,rear,right",
    "use_raw_post_processed": False,
    "image_size": IMAGE_SIZE,
    "num_workers": NUM_WORKERS,
    "batch_size": BATCH_SIZE,
    "val_ratio": 0.2,
    "test_ratio": 0.1,
    "stratify_by_camera": True,
    "seed": 42,
    "epochs": EPOCHS,
    "learning_rate": 0.001,
    "weight_decay": 0.0001,
    "model": {
        "name": "unet",
        "in_channels": 3,
        "base_channels": BASE_CHANNELS,
        "activation": "relu",
        "weight_init": "he_normal",
    },
    "loss": LOSS,
    "class_weights": [5.0, 15.0, 10.0],
    "dice_exclude_classes": [0],
    "checkpoint_interval": 4,
    "image_log_interval": 1,
    "debug_image_count": 8,
    "run_dir": str(RUN_DIR),
}

CONFIG_PATH.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
print(CONFIG_PATH.read_text(encoding="utf-8"))

## 7. Debug Dataset Overlay

In [ ]:
image_size_text = f"{IMAGE_SIZE[0]},{IMAGE_SIZE[1]}"
cmd = [
    "python", "src/segmentation/tools/debug_dataset.py",
    "--data-root", str(DATA_ROOT),
    "--manifest", str(MANIFEST),
    "--count", "8",
    "--image-size", image_size_text,
    "--out-dir", str(DEBUG_DIR),
]
subprocess.run(cmd, check=True)
print("debug overlays:", DEBUG_DIR)

## 8. Overfit Smoke Test

작은 sample 10개를 먼저 외워보게 해서 데이터 pair와 loss 흐름이 정상인지 확인한다.

In [ ]:
cmd = [
    "python", "-u", "src/segmentation/train.py",
    "--config", str(CONFIG_PATH),
    "--overfit-count", "10",
    "--epochs", "30",
    "--device", "auto",
]
subprocess.run(cmd, check=True, env=TRAIN_ENV)
print("overfit runs:", RUN_DIR)

## 9. Train

In [ ]:
cmd = [
    "python", "-u", "src/segmentation/train.py",
    "--config", str(CONFIG_PATH),
    "--device", "auto",
]
subprocess.run(cmd, check=True, env=TRAIN_ENV)
print("runs:", RUN_DIR)

## 10. TensorBoard

In [ ]:
from tensorboard import notebook

notebook.start("--logdir " + str(RUN_DIR))